## Conversation Graph

#### Review: Conversation Analytics describe how a conversation behaves from start to finish, not just message‑by‑message.

**Instead of asking:** “What is the intent of this message?”

**Conversation‑level insights ask:** “What happened across the entire conversation?”  
This is the difference between micro‑analysis (message‑level) and macro‑analysis (conversation‑level).

#### Flow:
Tweet_id = current tweet  
in_response_to_tweet_id = paretn tweet id to witch the current tweet_id is tweeting  
response_tweet_id = tweet id's replying to the current tweet id

#### Analysis: 
parent <-current <- childrens  
in_response_to_tweet_id <- Tweet_id <- response_tweet_id (may contain multiple responses)



#### Goal: Build conversational Graph

1. Clean response_tweet_id  
Convert "2987951,2987949" → [2987951, 2987949]

2. Build maps  
parent_map: tweet → parent

child_map: tweet → children

3. Find the root tweet  
Walk up the chain until you reach a tweet with no parent.

4. Assign conversation_id = root tweet  
This is exactly how chat logs are reconstructed.

In [8]:
# Import data
import pandas as pd

df = pd.read_csv("../data/twcs_convo_graph_ready.csv")
df.head(50)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,Clean_text,conversation_id,text_lemma,intent,confidence,is_low_conf,is_fallback,is_user,is_company,date_time
0,192624,161253,True,Wed Oct 04 13:59:33 +0000 2017,@161252 What's that egg website people talk about,192623,192625.0,what s that egg website people talk about,192625.0,s egg website people talk,refund_request,0.631970,False,False,True,False,2017-10-04 13:59:33+00:00
1,738238,296574,True,Fri Oct 06 18:29:06 +0000 2017,Why!🤷🏻‍♀️ #iOS11 @AppleSupport https://t.co/BX...,738237,NaN,why,738238.0,NaN,technical_problem,0.682013,False,False,True,False,2017-10-06 18:29:06+00:00
2,2414302,AppleSupport,False,Tue Nov 14 17:38:01 +0000 2017,@693975 We can assist you. We recommend updati...,2414303,2414304.0,we can assist you we recommend updating to ios...,2414304.0,assist recommend update io 11 1 1 haven t chan...,account_access,0.891190,False,False,False,True,2017-11-14 17:38:01+00:00
3,1793929,539096,True,Thu Oct 12 06:04:41 +0000 2017,@331912 @115955 Thats better than having an un...,1793928,1793930.0,thats better than having an unstable connectio...,1793930.0,s well have unstable connection drop 5 20 min,technical_problem,0.433034,True,True,True,False,2017-10-12 06:04:41+00:00
4,2088018,617376,True,Mon Nov 06 20:30:49 +0000 2017,@VirginAmerica is probably one of the best air...,2088017,NaN,is probably one of the best airlines i ve ever...,2088018.0,probably good airline ve experience,refund_request,0.427244,True,True,True,False,2017-11-06 20:30:49+00:00
5,1941217,577298,True,Sun Oct 29 23:13:33 +0000 2017,@AlaskaAir Phone system seems to be down,1941219,1941216.0,phone system seems to be down,1941216.0,phone system,account_access,0.466580,True,True,True,False,2017-10-29 23:13:33+00:00
6,863668,SpotifyCares,False,Tue Nov 21 05:56:34 +0000 2017,@325106 Cool! Can you DM us a screenshot of yo...,NaN,863669.0,cool can you dm us a screenshot of your bank s...,863669.0,cool dm screenshot bank statement rb,account_access,0.532173,True,True,False,True,2017-11-21 05:56:34+00:00
7,2206333,645101,True,Fri Nov 10 00:12:30 +0000 2017,@Uber_Support Reports are there are 68 Ubers w...,2206335,2206332.0,reports are there are 68 ubers waiting for pas...,2206332.0,report 68 uber wait passenger isn t work,unknown,0.939616,False,True,True,False,2017-11-10 00:12:30+00:00
8,1895823,AppleSupport,False,Wed Oct 18 12:33:00 +0000 2017,@565060 Since our Twitter support is available...,NaN,1895824.0,since our twitter support is available in engl...,1895824.0,twitter support available english help join,billing_issue,0.505595,True,True,False,True,2017-10-18 12:33:00+00:00
9,2744170,162033,True,Mon Nov 20 19:15:58 +0000 2017,@AmazonHelp Saw that but only after a labored ...,NaN,2744169.0,saw that but only after a labored hunt bigger ...,2744169.0,see labored hunt big point usb 2 0 expect expl...,unknown,0.746737,False,True,True,False,2017-11-20 19:15:58+00:00


### Step 1 — Normalize response_tweet_id

**python split explained:**
x.split(",")  
If x = "1,2,3,4"  
then:  
python: x.split(",")  
returns: ["1", "2", "3", "4"]  
It splits the string into a list of substrings.

In [4]:
df['response_tweet_id'] = df['response_tweet_id'].fillna("").astype(str)


In [21]:
# create a list of values for rows with more than one tweets

def parse_children(x):
    # Convert to string safely
    x = str(x)

    # Handle NaN or empty
    if x.lower() == "nan" or x.strip() == "":
        return []

    # Split by comma
    parts = x.split(",")

    # Convert each part to int if possible
    children = []
    for p in parts:
        p = p.strip()
        if p.isdigit():
            children.append(int(p))
    return children

df["response_list"] = df["response_tweet_id"].apply(parse_children)


### build the conversation graph

#### Step 1 — Build parent map

In [22]:
parent_map = dict(zip(df["tweet_id"], df["in_response_to_tweet_id"]))


#### Step 2 — Build child map

In [23]:
child_map = dict(zip(df["tweet_id"], df["response_list"]))
child_map

{192624: [],
 738238: [],
 2414302: [],
 1793929: [],
 2088018: [],
 1941217: [],
 863668: [],
 2206333: [],
 1895823: [],
 2744170: [],
 844825: [],
 2484393: [],
 1047538: [],
 960215: [],
 524920: [],
 57162: [],
 2573837: [],
 1820869: [],
 2180734: [],
 2972824: [],
 2724137: [],
 2585655: [],
 341050: [],
 1187784: [],
 1814543: [],
 697857: [],
 78291: [],
 2606719: [],
 2744480: [],
 647159: [],
 991947: [],
 17567: [],
 317275: [],
 1974474: [],
 163996: [],
 1960218: [],
 2474323: [],
 2798966: [],
 1715177: [],
 872288: [],
 2432781: [],
 1523325: [],
 2305368: [],
 1194176: [],
 1657619: [],
 2449598: [],
 2780106: [],
 2713852: [],
 1291: [],
 565872: [],
 530608: [],
 174953: [],
 1200719: [],
 953217: [],
 5748: [],
 2278301: [],
 963063: [],
 2103597: [],
 215127: [],
 1754653: [],
 970615: [],
 248949: [],
 1429569: [],
 2315949: [],
 2117991: [],
 1889585: [],
 2052625: [],
 954249: [],
 1212378: [],
 903259: [],
 1782593: [],
 359787: [],
 863703: [],
 1756216: [],
 

#### This random sample:
- Did NOT include any tweets that have replies
- Did NOT include any tweets that are replies
- Did NOT include any multi‑turn threads
- Did NOT include any conversation roots  
So our sample is basically: 5000 isolated tweets with no reply structure.  
— because the full dataset has 2.8 million tweets, and multi‑turn conversations are rare.
A random sample often misses them entirely.

## Fix

To build conversation graphs, you must use the full dataset, not the 5,000‑row sample.

#### OR

Instead of random sampling, we can use conversation-aware sampling:

#### Step 1 — Keep only tweets that have replies or parents
```
df_threaded = df[
    df["response_tweet_id"].notna() |
    df["in_response_to_tweet_id"].notna()
]
```
#### Step 2 — Sample from threaded tweets
```
df_small = df_threaded.sample(5000, random_state=42)
```
#### This guarantees:
- multi‑turn conversations
- parent→child chains
- child→parent chains
- real conversation threads


## Rebuild the conversation graph — using the full dataset

### STEP 1 — Load the FULL dataset (not the sample)

In [26]:
df = pd.read_csv("../data/twcs_chat_logs.csv")
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


### STEP 2 — Clean response_tweet_id properly

In [28]:
def parse_children(x):
    x = str(x)
    if x.lower() == "nan" or x.strip() == "":
        return []
    parts = x.split(",")
    children = []
    for p in parts:
        p = p.strip()
        if p.isdigit():
            children.append(int(p))
    return children

df["response_list"] = df["response_tweet_id"].apply(parse_children)
df["response_list"] 

0                         [2]
1                          []
2                         [1]
3                         [3]
4                         [4]
                  ...        
2811769                    []
2811770             [2987947]
2811771                    []
2811772                    []
2811773    [2987951, 2987949]
Name: response_list, Length: 2811774, dtype: object

### STEP 3 — Build parent and child maps

In [30]:
parent_map = dict(zip(df["tweet_id"], df["in_response_to_tweet_id"]))
child_map = dict(zip(df["tweet_id"], df["response_list"]))
child_map

{1: [2],
 2: [],
 3: [1],
 4: [3],
 5: [4],
 6: [5, 7],
 8: [9, 6, 10],
 11: [],
 12: [11, 13, 14],
 15: [12],
 16: [15],
 17: [16],
 18: [17],
 19: [],
 20: [19],
 21: [22, 23],
 22: [25],
 25: [26],
 26: [27],
 27: [],
 23: [],
 24: [21],
 28: [24],
 29: [28],
 30: [],
 31: [30],
 32: [],
 33: [32],
 34: [35],
 35: [37],
 37: [],
 36: [34],
 38: [],
 39: [38],
 40: [41],
 41: [43],
 43: [44],
 44: [45],
 45: [],
 42: [40],
 46: [42],
 47: [46],
 48: [47],
 49: [48],
 50: [],
 51: [50],
 52: [51],
 53: [52],
 54: [53],
 55: [54],
 56: [55],
 57: [56],
 58: [57],
 59: [58],
 60: [],
 61: [60],
 62: [],
 63: [62],
 64: [65],
 65: [],
 66: [64, 67],
 68: [],
 69: [68],
 70: [],
 71: [70, 72],
 73: [],
 74: [73],
 75: [74],
 76: [75],
 77: [],
 78: [77],
 79: [],
 80: [79],
 81: [80,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  

### STEP 4 — Find the root tweet of each conversation

In [31]:
def get_root(tweet_id):
    parent = parent_map.get(tweet_id)
    while parent and parent in parent_map:
        tweet_id = parent
        parent = parent_map.get(tweet_id)
    return tweet_id


### STEP 5 — Assign conversation_id

In [34]:
df["conversation_id"] = df["tweet_id"].apply(get_root)
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,response_list,conversation_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0,[2],8.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0,[],8.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0,[1],8.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0,[3],8.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0,[4],8.0


### STEP 6 — Compute conversation sizes

In [35]:
conv_sizes = df.groupby("conversation_id").size().sort_values(ascending=False)
conv_sizes.head(10)


conversation_id
625855.0     1390
776752.0     1060
37012.0       974
5965.0        802
88944.0       661
56184.0       651
2983022.0     650
615427.0      584
278169.0      551
2390546.0     534
dtype: int64

##### These are your real multi‑turn conversations.

### Option 2: Create a smaller dataset WITH conversations

**This guarantees:**  
- multi‑turn conversations
- reply chains
- real conversation threads

In [48]:
df_threaded = df[
    df["response_tweet_id"].notna() |
    df["in_response_to_tweet_id"].notna()
]

df_small = df_threaded.sample(5000, random_state=42)
df_small.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,response_list,conversation_id
160535,192624,161253,True,Wed Oct 04 13:59:33 +0000 2017,@161252 What's that egg website people talk about,192623,192625.0,[192623],192626.0
659248,738238,296574,True,Fri Oct 06 18:29:06 +0000 2017,Why!🤷🏻‍♀️ #iOS11 @AppleSupport https://t.co/BX...,738237,NaN,[738237],738238.0
2250310,2414302,AppleSupport,False,Tue Nov 14 17:38:01 +0000 2017,@693975 We can assist you. We recommend updati...,2414303,2414304.0,[2414303],2414304.0
1640680,1793929,539096,True,Thu Oct 12 06:04:41 +0000 2017,@331912 @115955 Thats better than having an un...,1793928,1793930.0,[1793928],1793933.0
1933623,2088018,617376,True,Mon Nov 06 20:30:49 +0000 2017,@VirginAmerica is probably one of the best air...,2088017,NaN,[2088017],2088018.0


### Step 7 — Analyze Long Conversations

Let’s extract the top 10 longest conversations:

In [53]:
top_conversations = conv_sizes.head(10).index.tolist()
top_conversations

[625855.0,
 776752.0,
 37012.0,
 5965.0,
 88944.0,
 56184.0,
 2983022.0,
 615427.0,
 278169.0,
 2390546.0]

### Step 7 — inspect one conversation:

In [54]:
df[df["conversation_id"] == top_conversations[0]].sort_values("created_at")


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,response_list,conversation_id
1461711,1606169,ATVIAssist,False,Fri Nov 10 00:15:31 +0000 2017,"@492555 Good afternoon, sorry for the delay, a...",NaN,625835.0,[],625855.0
1461707,1606167,ATVIAssist,False,Fri Nov 10 00:21:55 +0000 2017,"@493399 Good afternoon, sorry for the delay, a...",NaN,625833.0,[],625855.0
1461705,1606166,ATVIAssist,False,Fri Nov 10 01:11:28 +0000 2017,"@493398 Good afternoon, sorry for the delay, a...",NaN,625832.0,[],625855.0
1461709,1606168,ATVIAssist,False,Fri Nov 10 01:50:53 +0000 2017,@493400 Hi there! If you are still having conn...,NaN,625834.0,[],625855.0
1461713,1606170,ATVIAssist,False,Fri Nov 10 02:46:52 +0000 2017,@493401 Hey there! It is automatically used wh...,NaN,625836.0,[],625855.0
...,...,...,...,...,...,...,...,...,...
1461748,1606187,493419,True,Wed Nov 15 19:47:06 +0000 2017,@493418 @ATVIAssist @115766 😂,1606188,625850.0,[1606188],625855.0
1461749,1606188,493418,True,Wed Nov 15 21:36:25 +0000 2017,@493419 @ATVIAssist @115766 Amateurs pal amate...,1606189,1606187.0,[1606189],625855.0
1461750,1606189,493419,True,Wed Nov 15 22:34:37 +0000 2017,"@493418 @ATVIAssist @115766 Bottons! Anyway, I...","1606190,1606191",1606188.0,"[1606190, 1606191]",625855.0
1461752,1606191,493418,True,Wed Nov 15 22:38:26 +0000 2017,"@493419 @ATVIAssist @115766 True pal lol, telt...",NaN,1606189.0,[],625855.0


### A. Conversation Length Insight
This conversation has 1390 messages — extremely long.

#### This indicates:
- high customer frustration
- unresolved issue
- repeated troubleshooting
- possible outage
- high support load
- multi‑user involvement

#### Intent Patterns
Run:
```
df[df["conversation_id"] == 625855.0]["intent"].value_counts()
```
**This will show:**
- which intents dominate long conversations
- which intents cause escalation
- which intents need better automation

#### Fallback & Low Confidence
Run:
```
df[df["conversation_id"] == 625855.0]["is_fallback"].mean()
df[df["conversation_id"] == 625855.0]["is_low_conf"].mean()
```
If fallback is high → bot confusion  
If low confidence is high → intent taxonomy issues  

#### User vs Company Behavior
Run:
```
df[df["conversation_id"] == 625855.0]["is_user"].value_counts(normalize=True)
```
**This shows:**
- how much the customer is talking
- how much the company is responding  
whether the conversation is balanced or one‑sided

#### Time Analysis
Run:
```
df[df["conversation_id"] == 625855.0]["created_at"].min()
df[df["conversation_id"] == 625855.0]["created_at"].max()
```
**This shows:**
- how many days the conversation lasted
- whether it escalated over time

### Intent distribution inside this conversation (apply to the sample df_small)

In [56]:
# Create a simple intent taxonomy
intents = [
    "billing_issue",
    "technical_problem",
    "account_access",
    "order_status",
    "refund_request",
    "general_query",
    "smalltalk",
    "unknown"
]

# Simulate intent labels
import numpy as np
df_small["intent"] = np.random.choice(intents, size=len(df_small))

# Simulate confidence scores
df_small["confidence"] = np.random.uniform(0.40, 0.99, size=len(df_small))

# Define fallback logic
low_conf_threshold = 0.6

df_small["is_low_conf"] = df_small["confidence"] < low_conf_threshold
df_small["is_fallback"] = (df_small["intent"] == "unknown") | df_small["is_low_conf"]

# Classifying inbound for analytics¶
df_small['is_user'] = df_small["inbound"] 
df_small["is_company"] = ~df_small["inbound"] 

df_small.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,response_list,conversation_id,intent,confidence,is_low_conf,is_fallback,is_user,is_company
160535,192624,161253,True,Wed Oct 04 13:59:33 +0000 2017,@161252 What's that egg website people talk about,192623,192625.0,[192623],192626.0,refund_request,0.587868,True,True,True,False
659248,738238,296574,True,Fri Oct 06 18:29:06 +0000 2017,Why!🤷🏻‍♀️ #iOS11 @AppleSupport https://t.co/BX...,738237,NaN,[738237],738238.0,unknown,0.579054,True,True,True,False
2250310,2414302,AppleSupport,False,Tue Nov 14 17:38:01 +0000 2017,@693975 We can assist you. We recommend updati...,2414303,2414304.0,[2414303],2414304.0,billing_issue,0.605361,False,False,False,True
1640680,1793929,539096,True,Thu Oct 12 06:04:41 +0000 2017,@331912 @115955 Thats better than having an un...,1793928,1793930.0,[1793928],1793933.0,order_status,0.984484,False,False,True,False
1933623,2088018,617376,True,Mon Nov 06 20:30:49 +0000 2017,@VirginAmerica is probably one of the best air...,2088017,NaN,[2088017],2088018.0,technical_problem,0.735044,False,False,True,False


### Calculate top conversations int he sample

In [63]:
convo_size_sample = df_small.groupby("conversation_id").size().sort_values(ascending=False)
convo_size_sample

conversation_id
37012.0      5
5965.0       4
625855.0     3
1860388.0    3
808639.0     2
            ..
2985825.0    1
2986344.0    1
2986514.0    1
2986906.0    1
2599031.0    1
Length: 4956, dtype: int64

In [59]:
top_convo = convo_size_sample.head(10).index.tolist()
top_convo

[37012.0,
 5965.0,
 625855.0,
 1860388.0,
 808639.0,
 1289937.0,
 355541.0,
 1343823.0,
 409.0,
 35977.0]

#### longest conversation in the sample
```
conversation_id = 37012.0
size = 5 messages
```
This is a short multi‑turn conversation, but still useful for analysis.

### Intent distribution inside this conversation

In [71]:
df_small[df_small["conversation_id"] == top_convo[0]]["intent"].value_counts()


intent
smalltalk         2
refund_request    1
billing_issue     1
order_status      1
Name: count, dtype: int64

### Interpretation
**This is a mixed‑intent conversation, which is extremely common in customer support:**
- Customer starts with smalltalk
- Then asks about refund
- Then mentions billing
- Then asks about order status

**This is a multi‑intent escalation pattern, meaning:***
- The customer is trying multiple ways to get help
- The issue may not be resolved quickly
- The bot or agent may be misunderstanding the intent
- The customer is switching topics to get attention  
This is a classic frustration pattern.

### Fallback rate inside this conversation

In [67]:
fallback_rate = df_small[df_small["conversation_id"] == top_convo[0]]["is_fallback"].mean()
fallback_rate

np.float64(0.6)

#### Interpretation
A 60% fallback rate is very high.

**This means:**
- The bot misunderstood the customer
- The bot didn’t know which intent to assign
- The bot kept saying “I didn’t understand”
- The conversation likely escalated because of confusion  
High fallback rate = poor NLU performance.


### Low confidence rate inside this conversation

In [62]:
df_small[df_small["conversation_id"] == top_convo[0]]["is_low_conf"].mean()


np.float64(0.6)

#### Interpretation
This is also extremely high.

**It means:**
- The model was unsure about the intent
- The model probably guessed wrong
- The model may have assigned incorrect intents
- The conversation became messy and multi‑intent  
High low‑confidence = intent taxonomy needs improvement.

## Summary:
What this conversation tells us 

This conversation is a perfect example of:  
✔ Multi‑intent escalation  
Customer jumps between refund → billing → order status.

✔ High bot confusion  
Fallback = 60%  
Low confidence = 60%

✔ Poor intent recognition  
Smalltalk appears twice — likely misclassified.

✔ Training data opportunity  
We need more examples for:  
- refund_request
- billing_issue
- order_status

✔ Taxonomy improvement  
These intents may overlap or be poorly defined.

In [65]:
### Save the sample
# save sample 
df_small.to_csv("../data/twcs_multiturn_sample.csv", index=False)

### Summary Cell

In [72]:
print("Total messages:", len(df_small))
print("Total conversations:", df_small['conversation_id'].nunique())
print("Fallback rate:", fallback_rate)
print("Top intents:", df_small["intent"].value_counts().head(5).to_dict())


Total messages: 5000
Total conversations: 4956
Fallback rate: 0.6
Top intents: {'unknown': 650, 'billing_issue': 641, 'order_status': 631, 'technical_problem': 626, 'account_access': 622}


Intents with highest low confidence: 223


In [85]:
top_low_conf = df_small.groupby('intent')['is_low_conf'].sum().sort_values(ascending=False)
top_low_conf.index[0]

'technical_problem'

In [93]:
print("Intents with highest low confidence -", top_low_conf.index[0] + ":" , top_low_conf.iloc[0])

Intents with highest low confidence - technical_problem: 223
